# CUSUM and EWMA for Anomaly Detection

This notebook implements **CUSUM (Cumulative Sum)** and **EWMA (Exponentially Weighted Moving Average)** control-chart anomaly detectors for SWaT per-PLC (per-stage) data. 

**Why CUSUM and EWMA for ICS?**
These methods are the lightweight statistical floor baseline in our 5-model comparison. They replace generic tabular baselines like Isolation Forest because they are specifically designed for continuous physical process and sensor monitoring:
- **EWMA** is excellent at detecting *gradual drift* in continuous variables by exponentially smoothing out high-frequency noise.
- **CUSUM** accumulates small deviations over time, making it highly sensitive to *abrupt but subtle shifts* in the process mean.
- **Extremely Cheap**: Both require minimal memory and sub-millisecond compute per tag, making them ideal for edge deployment (e.g., directly on PLCs).

**Note on Preprocessing**: CUSUM and EWMA are designed for *continuous* process variables. Therefore, we explicitly exclude binary actuator columns (which only take values 0 or 1). This is consistent with the PASAD notebook's methodology.

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, f1_score, precision_score, recall_score, roc_curve

# Top parameter cell
STAGE_NUM = 1 # Change this to 1-6
LAMBDA_EWMA = 0.2 # EWMA smoothing factor
CUSUM_K_FACTOR = 0.5 # k = k_factor * normal std
CUSUM_H_FACTOR = 5.0 # h = h_factor * normal std
L_EWMA = 3.0 # EWMA Control limits (L-sigma)

# Paths
NORMAL_DATA_PATH = f'dataset/normal_stage{STAGE_NUM}.csv'
ATTACK_DATA_PATH = f'dataset/attack_stage{STAGE_NUM}.csv'
os.makedirs('models', exist_ok=True)
MODEL_SAVE_PATH = f'models/cusum_ewma_stage{STAGE_NUM}.json'

In [ ]:
def get_continuous_columns(df):
    """Select CONTINUOUS sensor columns only, exclude binary actuators."""
    cols = df.columns.tolist()
    if 'Timestamp' in cols:
        cols.remove('Timestamp')
    if 'Normal/Attack' in cols:
        cols.remove('Normal/Attack')
    if 'attack' in cols:
        cols.remove('attack')
    
    continuous_cols = []
    for col in cols:
        # Simple heuristic: if unique values > 3, it's likely a continuous sensor
        if df[col].nunique() > 3:
            continuous_cols.append(col)
    return continuous_cols

def load_and_preprocess(filepath, is_attack=False):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    
    if 'Timestamp' in df.columns:
        # Avoid full datetime parsing if it's too slow, but sorting is required
        df = df.sort_values('Timestamp').reset_index(drop=True)
        
    # Coerce numeric
    cols_to_convert = [c for c in df.columns if c not in ['Timestamp', 'Normal/Attack', 'attack']]
    for c in cols_to_convert:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    # Forward-fill NaNs then backward fill
    df = df.ffill().bfill()
    return df

print(f"Loading Stage {STAGE_NUM} data...")
df_normal = load_and_preprocess(NORMAL_DATA_PATH, is_attack=False)
df_attack = load_and_preprocess(ATTACK_DATA_PATH, is_attack=True)

continuous_cols = get_continuous_columns(df_normal)
print(f"Selected {len(continuous_cols)} continuous columns: {continuous_cols}")

## Per-Tag Baseline Statistics
Fit on `normal_stage{N}` ONLY: Mean ($\mu$) and standard deviation ($\sigma$) of each tag during normal operation. This provides the statistical foundation for the control charts.

In [ ]:
stats = {}
for col in continuous_cols:
    mu = df_normal[col].mean()
    sigma = df_normal[col].std()
    if sigma == 0 or pd.isna(sigma):
        sigma = 1e-6 # prevent division by zero
    stats[col] = {
        'mu': mu,
        'sigma': sigma,
        'k': CUSUM_K_FACTOR * sigma,
        'h': CUSUM_H_FACTOR * sigma,
        'ewma_limit': L_EWMA * sigma * np.sqrt(LAMBDA_EWMA / (2 - LAMBDA_EWMA))
    }
print("Baseline statistics computed for normal data.")

In [ ]:
def run_ewma(df, stats, cols, lambda_ewma):
    scores = {}
    flags = {}
    
    for col in cols:
        x = df[col].values
        mu = stats[col]['mu']
        limit = stats[col]['ewma_limit']
        
        z = np.zeros(len(x))
        z[0] = mu
        # Vectorized or simple iterative implementation
        for t in range(1, len(x)):
            z[t] = lambda_ewma * x[t] + (1 - lambda_ewma) * z[t-1]
            
        # Anomaly score: distance from mean normalized by control limit width
        dev = np.abs(z - mu)
        score = dev / (limit + 1e-9)
        flag = score > 1.0
        
        scores[col] = score
        flags[col] = flag
        
    return scores, flags

print("Running EWMA on attack data...")
ewma_scores, ewma_flags = run_ewma(df_attack, stats, continuous_cols, LAMBDA_EWMA)

In [ ]:
def run_cusum(df, stats, cols):
    scores = {}
    flags = {}
    
    for col in cols:
        x = df[col].values
        mu = stats[col]['mu']
        k = stats[col]['k']
        h = stats[col]['h']
        
        s_high = np.zeros(len(x))
        s_low = np.zeros(len(x))
        
        for t in range(1, len(x)):
            s_high[t] = max(0, s_high[t-1] + (x[t] - mu - k))
            s_low[t] = max(0, s_low[t-1] + (mu - k - x[t]))
            
        # Anomaly score: max of high and low CUSUM, normalized by h
        max_s = np.maximum(s_high, s_low)
        score = max_s / (h + 1e-9)
        flag = score > 1.0
        
        scores[col] = score
        flags[col] = flag
        
    return scores, flags

print("Running CUSUM on attack data...")
cusum_scores, cusum_flags = run_cusum(df_attack, stats, continuous_cols)

In [ ]:
def fuse_scores(scores_dict, df_normal, stats_func_or_dict, method_runner, cols):
    """
    Same pattern as PASAD: min-max normalize each tag's score on normal data,
    then max pool across tags into one per-timestep stage-level score.
    """
    norm_scores, _ = method_runner(df_normal)
    
    norm_min = {}
    norm_max = {}
    for col in cols:
        norm_min[col] = np.min(norm_scores[col])
        norm_max[col] = np.max(norm_scores[col])
        if norm_max[col] == norm_min[col]:
            norm_max[col] = norm_min[col] + 1e-6
            
    fused_score = np.zeros(len(list(scores_dict.values())[0]))
    for col in cols:
        s = scores_dict[col]
        s_norm = (s - norm_min[col]) / (norm_max[col] - norm_min[col])
        s_norm = np.clip(s_norm, 0, None)
        fused_score = np.maximum(fused_score, s_norm)
        
    return fused_score

print("Calibrating fusion on normal data...")
def run_ewma_wrapper(df): return run_ewma(df, stats, continuous_cols, LAMBDA_EWMA)
def run_cusum_wrapper(df): return run_cusum(df, stats, continuous_cols)

fused_ewma_score = fuse_scores(ewma_scores, df_normal, stats, run_ewma_wrapper, continuous_cols)
fused_cusum_score = fuse_scores(cusum_scores, df_normal, stats, run_cusum_wrapper, continuous_cols)

In [ ]:
def evaluate_model(y_true, scores, name, is_stage6=False):
    print(f"\n{'='*40}")
    print(f"Evaluation for {name}")
    print(f"{'='*40}")
    
    if is_stage6:
        print("Stage 6 lacks ground truth attacks. Plotting fused score time series.")
        plt.figure(figsize=(12, 4))
        plt.plot(scores, label=f"{name} Fused Score")
        plt.title(f"{name} Fused Anomaly Score - Stage 6")
        plt.xlabel("Time")
        plt.ylabel("Score")
        plt.legend()
        plt.show()
        print(f"Summary Stats: Mean={np.mean(scores):.4f}, Max={np.max(scores):.4f}")
        return None
        
    # Because we max-pooled min-max normalized scores from normal data,
    # the normal period max is strictly 1.0. Therefore, threshold is 1.0.
    threshold = 1.0 
    y_pred = (scores > threshold).astype(int)
    
    auc = roc_auc_score(y_true, scores)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    print(f"ROC-AUC: {auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"FPR: {fpr:.4f}")
    print(f"FNR: {fnr:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    return {'auc': auc, 'f1': f1, 'prec': prec, 'rec': rec, 'fpr': fpr, 'fnr': fnr}

def per_tag_contribution(y_true, flags_dict, name):
    print(f"\n{name} - Per-Tag Contribution (Firing during True Positives):")
    tp_mask = (y_true == 1)
    if not np.any(tp_mask):
        print("No true positives in this dataset.")
        return
        
    contributions = {}
    for col, flags in flags_dict.items():
        tp_fires = np.sum(flags[tp_mask])
        if tp_fires > 0:
            contributions[col] = tp_fires
            
    sorted_cont = sorted(contributions.items(), key=lambda x: x[1], reverse=True)
    for k, v in sorted_cont:
        print(f"{k}: {v} fires")

def measure_latency(df, stats, cols, method_name, method_runner):
    sub_df = df.iloc[:1000] if len(df) > 1000 else df
    start_time = time.perf_counter()
    method_runner(sub_df)
    end_time = time.perf_counter()
    
    total_ms = (end_time - start_time) * 1000
    latency_per_step = total_ms / len(sub_df)
    latency_per_tag = latency_per_step / len(cols)
    print(f"\n{method_name} Inference Latency:")
    print(f"- Mean latency per timestep: {latency_per_step:.4f} ms")
    print(f"- Mean latency per tag per timestep: {latency_per_tag:.4f} ms")

if 'attack' in df_attack.columns:
    y_true = df_attack['attack'].values
elif 'Normal/Attack' in df_attack.columns:
    y_true = (df_attack['Normal/Attack'] != 'Normal').astype(int)
else:
    y_true = np.zeros(len(df_attack))

is_stage6 = (STAGE_NUM == 6)

metrics_ewma = evaluate_model(y_true, fused_ewma_score, "EWMA", is_stage6)
metrics_cusum = evaluate_model(y_true, fused_cusum_score, "CUSUM", is_stage6)

if not is_stage6:
    per_tag_contribution(y_true, ewma_flags, "EWMA")
    per_tag_contribution(y_true, cusum_flags, "CUSUM")

measure_latency(df_attack, stats, continuous_cols, "EWMA", run_ewma_wrapper)
measure_latency(df_attack, stats, continuous_cols, "CUSUM", run_cusum_wrapper)

In [ ]:
with open(MODEL_SAVE_PATH, 'w') as f:
    json.dump(stats, f, indent=4)
print(f"Saved calibrated per-tag statistics to {MODEL_SAVE_PATH}")

In [ ]:
# Final cell looping STAGE_NUM 1-6
all_metrics = []

for s in range(1, 7):
    norm_path = f'dataset/normal_stage{s}.csv'
    attk_path = f'dataset/attack_stage{s}.csv'
    
    if not os.path.exists(norm_path) or not os.path.exists(attk_path):
        print(f"Stage {s} data not found, skipping loop.")
        continue
        
    df_n = load_and_preprocess(norm_path, is_attack=False)
    df_a = load_and_preprocess(attk_path, is_attack=True)
    c_cols = get_continuous_columns(df_n)
    
    s_stats = {}
    for col in c_cols:
        mu = df_n[col].mean()
        sigma = df_n[col].std()
        if sigma == 0 or pd.isna(sigma): sigma = 1e-6
        s_stats[col] = {
            'mu': mu, 'sigma': sigma,
            'k': CUSUM_K_FACTOR * sigma, 'h': CUSUM_H_FACTOR * sigma,
            'ewma_limit': L_EWMA * sigma * np.sqrt(LAMBDA_EWMA / (2 - LAMBDA_EWMA))
        }
    
    # EWMA
    e_norm, _ = run_ewma(df_n, s_stats, c_cols, LAMBDA_EWMA)
    e_attk, _ = run_ewma(df_a, s_stats, c_cols, LAMBDA_EWMA)
    n_min, n_max = {}, {}
    for col in c_cols:
        n_min[col] = np.min(e_norm[col])
        n_max[col] = max(np.max(e_norm[col]), n_min[col]+1e-6)
    e_fused = np.zeros(len(df_a))
    for col in c_cols:
        s_norm = np.clip((e_attk[col] - n_min[col]) / (n_max[col] - n_min[col]), 0, None)
        e_fused = np.maximum(e_fused, s_norm)
        
    # CUSUM
    c_norm, _ = run_cusum(df_n, s_stats, c_cols)
    c_attk, _ = run_cusum(df_a, s_stats, c_cols)
    c_min, c_max = {}, {}
    for col in c_cols:
        c_min[col] = np.min(c_norm[col])
        c_max[col] = max(np.max(c_norm[col]), c_min[col]+1e-6)
    c_fused = np.zeros(len(df_a))
    for col in c_cols:
        s_norm = np.clip((c_attk[col] - c_min[col]) / (c_max[col] - c_min[col]), 0, None)
        c_fused = np.maximum(c_fused, s_norm)
        
    y_t = df_a['attack'].values if 'attack' in df_a.columns else (df_a['Normal/Attack'] != 'Normal').astype(int)
    
    if s == 6:
        all_metrics.append({'Stage': s, 'Method': 'EWMA', 'AUC': None, 'F1': None})
        all_metrics.append({'Stage': s, 'Method': 'CUSUM', 'AUC': None, 'F1': None})
    else:
        e_pred = (e_fused > 1.0).astype(int)
        c_pred = (c_fused > 1.0).astype(int)
        
        all_metrics.append({
            'Stage': s, 'Method': 'EWMA', 
            'AUC': roc_auc_score(y_t, e_fused) if len(np.unique(y_t)) > 1 else None, 
            'F1': f1_score(y_t, e_pred, zero_division=0),
            'FPR': sum((e_pred==1)&(y_t==0))/sum(y_t==0) if sum(y_t==0)>0 else 0,
            'FNR': sum((e_pred==0)&(y_t==1))/sum(y_t==1) if sum(y_t==1)>0 else 0
        })
        all_metrics.append({
            'Stage': s, 'Method': 'CUSUM', 
            'AUC': roc_auc_score(y_t, c_fused) if len(np.unique(y_t)) > 1 else None, 
            'F1': f1_score(y_t, c_pred, zero_division=0),
            'FPR': sum((c_pred==1)&(y_t==0))/sum(y_t==0) if sum(y_t==0)>0 else 0,
            'FNR': sum((c_pred==0)&(y_t==1))/sum(y_t==1) if sum(y_t==1)>0 else 0
        })

summary_df = pd.DataFrame(all_metrics)
try:
    display(summary_df)
except NameError:
    print(summary_df)